# S07: release only with the right evidence

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cybertide-solutions/ai-platform-architect-labs/blob/main/notebooks/06_release_engineering.ipynb)

**Start:** run the first code cell. It automatically downloads the supporting files in a fresh Colab runtime. No ZIP upload, local installation or pasted setup code is required.

70-minute lab. A release consists of code, model, prompt, data, permissions and configuration. Demonstrate a candidate regression and a rollback; distinguish this software gate from AI acceptance.

In [ ]:
# Automatic lab setup: run this cell first. No ZIP upload or manual clone is needed.
from pathlib import Path
import os, sys, json, tempfile, subprocess, shutil

def prepare_lab_files(colab_home=Path('/content')):
    required = ('engine/__init__.py', 'engine/knowledge.py', 'data/policies.json', 'data/orders.json')
    def ready(path):
        return all((path / name).is_file() for name in required)
    try:
        import google.colab
        in_colab = True
    except ImportError:
        in_colab = False
    target = colab_home / 'ai-platform-architect-labs'
    candidates = ([target] if in_colab else []) + [Path.cwd(), Path.cwd().parent]
    root = next((path for path in candidates if ready(path)), None)
    if root is None and in_colab:
        if target.exists():
            raise RuntimeError('An incomplete course folder already exists. Save your work, then disconnect and delete this Colab runtime and reopen the notebook.')
        print('Downloading the course code and data from GitHub...')
        colab_home.mkdir(parents=True, exist_ok=True)
        staging = Path(tempfile.mkdtemp(prefix='course-download-', dir=colab_home))
        try:
            result = subprocess.run(
                ['git', 'clone', '--quiet', '--depth', '1', '--branch', 'main',
                 'https://github.com/cybertide-solutions/ai-platform-architect-labs.git', str(staging / 'repo')],
                capture_output=True, text=True, timeout=120)
            if result.returncode != 0 or not ready(staging / 'repo'):
                raise RuntimeError('Course download failed. Check the runtime internet connection and rerun this cell. Your existing work was not changed.')
            (staging / 'repo').rename(target)
            root = target
        except subprocess.TimeoutExpired:
            raise RuntimeError('Course download timed out. Rerun this cell to try again.') from None
        finally:
            shutil.rmtree(staging, ignore_errors=True)
    if root is None:
        raise RuntimeError('Local Jupyter: open this notebook from the extracted course folder. In Colab, use the README Open in Colab button.')
    root = root.resolve()
    os.chdir(root)
    if str(root) not in sys.path:
        sys.path.insert(0, str(root))
    print('Lab files ready. Continue with the cells below.')
    return root

ROOT = prepare_lab_files()
# End automatic setup
from engine import read_data
from engine.identity import BUYER, FINANCE, BEACON
from engine.knowledge import Index, evidence_errors
from engine.purchasing import Purchases, query_errors
from engine.platform import Platform
from engine.provider import ModelClient
# Deliberately opt in. No credential is required for software rehearsal.
# Colab only: enable after saving these names in Colab Secrets and granting access.
USE_COLAB_SECRETS=False
if USE_COLAB_SECRETS:
    from google.colab import userdata
    for name in ['CHAT_URL','CHAT_MODEL','CHAT_KEY','EMBED_URL','EMBED_MODEL','EMBED_KEY']:
        os.environ[name]=userdata.get(name)
    os.environ['LIVE_MODE']='1'
LIVE=os.environ.get('LIVE_MODE')=='1'
client=ModelClient.environment() if LIVE else None
print('LIVE MODEL' if LIVE else 'SOFTWARE REHEARSAL: no generated AI answers or AI quality score')

**0–20 min | Establish a baseline and a bad candidate.** Remove the current payment section in a new index version. Evaluate both against the same frozen cases. The candidate should fail relevant retrieval cases. Rollback selects a known version rather than editing historical evidence.

In [ ]:
from engine.evaluate import retrieval_suite,release_gate
ix=Index();docs=read_data('policies.json');ix.build('v1',docs)
ix.build('candidate', [d for d in docs if d['id']!='A-TERMS'])
baseline=release_gate(retrieval_suite(ix,'v1'));candidate=release_gate(retrieval_suite(ix,'candidate'))
print('baseline',baseline);print('candidate',candidate)
assert baseline['release'] and not candidate['release']
active='candidate' if candidate['release'] else 'v1'
assert active=='v1'
print('Active version:',active)

**20–40 min | Require complete review.** Structural and access tests are necessary. For generated answers, all nine labelled cases require correct/supported/complete human review. Unreviewed values cannot count as passes. Do not assume an LLM judge is calibrated; compare it with independent human labels first.

In [ ]:
manifest={'code':'record commit or ZIP hash','model':client.model if LIVE else None,
          'prompt':'policy-v1','index':'v1','permissions':'fictional manifest v1','runtime':'record Python version'}
path=Path('outputs/mini-project-review.json')
review=json.loads(path.read_text()) if path.exists() else []
required={t['id'] for t in read_data('answer_cases.json')}
semantic_ready=(len(review)==len(required) and {r['case'] for r in review}==required
                and all(r['actual']['status'] in ('answered','insufficient') and
                        all(r.get(k) is True for k in ['human_correct','human_supported','human_complete']) for r in review))
print('Manifest',manifest)
print('Software baseline acceptable:',baseline['release'])
print('Semantic review complete and acceptable:',semantic_ready)
print('Classroom AI release decision:',bool(manifest['model']) and baseline['release'] and semantic_ready)

**40–60 min | Write a canary plan.** Define the exposure unit, eligible traffic, minimum evidence, failure slices and rollback signal. Do not average away access violations. This notebook performs a local version switch, not a live traffic canary. Explain why a provider/model change requires fresh evaluation even behind a compatible API.

**60–70 min | Deliver:** two evaluation reports; manifest; rejection reason; rollback demonstration; outstanding semantic evidence; one proposed production sample-size improvement. The small supplied case set is a teaching instrument, not proof of general safety or production quality.

In [ ]:
ix.close()